In [2]:
import sys
import os
import shutil
import stat

# Triton / ptxas permission fix for Kaggle read-only NVIDIA utility script.
src_bin = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin"
dst_bin = "/tmp/triton_nvidia_bin"

if os.path.exists(src_bin):
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)

    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, 0o755)
            print(fp, oct(os.stat(fp).st_mode)[-3:])

    ptxas_path = os.path.join(dst_bin, "ptxas")
    ptxas_blackwell_path = os.path.join(dst_bin, "ptxas-blackwell")

    if os.path.exists(ptxas_path):
        os.environ["TRITON_PTXAS_PATH"] = ptxas_path

    if os.path.exists(ptxas_blackwell_path):
        os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = ptxas_blackwell_path

    # Put writable binaries first on PATH.
    os.environ["PATH"] = dst_bin + ":" + os.environ.get("PATH", "")

    # Use writable Triton cache.
    os.environ["TRITON_CACHE_DIR"] = "/tmp/triton-cache"
    os.makedirs(os.environ["TRITON_CACHE_DIR"], exist_ok=True)

    try:
        import triton.backends.nvidia as nv_backend

        # Point the backend module location toward the writable copy.
        nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")

        import triton.backends.nvidia.compiler as nv_compiler

        def _safe_get_ptxas_version(*args, **kwargs):
            return (12, 9, 0)

        nv_compiler.get_ptxas_version = _safe_get_ptxas_version

        print("Triton ptxas fix applied ✓")
        print("TRITON_PTXAS_PATH:", os.environ.get("TRITON_PTXAS_PATH"))
        print("TRITON_PTXAS_BLACKWELL_PATH:", os.environ.get("TRITON_PTXAS_BLACKWELL_PATH"))
        print("TRITON_CACHE_DIR:", os.environ.get("TRITON_CACHE_DIR"))

    except Exception as e:
        print(f"Triton ptxas partial fix applied, but compiler monkeypatch failed: {e}")
else:
    print("Triton NVIDIA bin folder not found — continuing without fix")

/tmp/triton_nvidia_bin/cuobjdump 755
/tmp/triton_nvidia_bin/ptxas-blackwell 755
/tmp/triton_nvidia_bin/ptxas 755
/tmp/triton_nvidia_bin/nvdisasm 755
Triton ptxas fix applied ✓
TRITON_PTXAS_PATH: /tmp/triton_nvidia_bin/ptxas
TRITON_PTXAS_BLACKWELL_PATH: /tmp/triton_nvidia_bin/ptxas-blackwell
TRITON_CACHE_DIR: /tmp/triton-cache


In [3]:
import os
import re
import json
import glob
import zipfile
import random
import warnings
from pathlib import Path

import pandas as pd
import torch

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# Competition/model settings
OUTPUT_DIR = "/kaggle/working"
ADAPTER_DIR = "/kaggle/working/final_adapter"
SUBMISSION_ZIP = "/kaggle/working/submission.zip"
LORA_RANK = 32  # competition maximum is 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Training settings. Tune these after the pipeline works.
MAX_TRAIN_ROWS = 2500      # increase toward len(train_df) after first successful run
VALID_FRACTION = 0.08
MAX_LENGTH = 2048          # larger may improve fit but uses more memory
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
WARMUP_RATIO = 0.03

# Optional validation smoke test after training.
RUN_SMOKE_VALIDATION = True
SMOKE_VALIDATION_ROWS = 16
MAX_NEW_TOKENS = 256

assert LORA_RANK <= 32, "LoRA rank must be <= 32 for this competition."
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())


CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
bf16 supported: True


In [4]:
# Find and inspect train/test files

def find_file(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under /kaggle/input")
    # Prefer the competition folder if present.
    matches = sorted(matches, key=lambda p: ("nvidia-nemotron" not in p, len(p)))
    return matches[0]

TRAIN_PATH = find_file("train.csv")
TEST_PATH = find_file("test.csv")
print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH:", TEST_PATH)

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
print("train columns:", train_df.columns.tolist())
display(train_df.head())

required = {"prompt", "answer"}
missing = required - set(train_df.columns)
assert not missing, f"Missing required train columns: {missing}"


TRAIN_PATH: /kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv
TEST_PATH: /kaggle/input/nvidia-nemotron-3-reasoning-challenge/test.csv
train shape: (9500, 3)
test shape: (3, 2)
train columns: ['id', 'prompt', 'answer']


,id,prompt,answer
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011
2,00189f6a,"In Alice's Wonderland, secret encryption rules...",cat imagines book
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII
4,001c63cb,"In Alice's Wonderland, secret encryption rules...",wizard creates secret


In [5]:
# Train/validation split
# We hold out some public train rows for a smoke check. This is not the hidden leaderboard,
# but it helps catch broken formatting and obvious regressions.

from sklearn.model_selection import train_test_split

work_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
if MAX_TRAIN_ROWS is not None:
    work_df = work_df.head(min(MAX_TRAIN_ROWS, len(work_df))).copy()

sft_train_df, sft_valid_df = train_test_split(
    work_df,
    test_size=VALID_FRACTION,
    random_state=SEED,
    shuffle=True,
)

sft_train_df = sft_train_df.reset_index(drop=True)
sft_valid_df = sft_valid_df.reset_index(drop=True)

print("SFT train rows:", len(sft_train_df))
print("SFT valid rows:", len(sft_valid_df))


SFT train rows: 2300
SFT valid rows: 200


In [6]:
# Prompt/answer formatting
# The hidden evaluator extracts final answers, prioritizing \boxed{...}. So this training
# format teaches the adapter to produce boxed final answers consistently.

SYSTEM_HINT = (
    "You are solving Alice's Wonderland rule-discovery puzzles. "
    "Infer the hidden transformation from the examples and answer the final query. "
    "End with the final answer in \\boxed{...}."
)

def clean_answer(x):
    return str(x).strip()

def build_user_prompt(prompt):
    # Plain instruction format avoids relying on a tokenizer-specific chat template.
    return (
        f"Instruction:\n{SYSTEM_HINT}\n\n"
        f"Puzzle:\n{str(prompt).strip()}\n\n"
        "Response:\n"
    )

def build_assistant_response(answer):
    # Answer-focused SFT: avoid fabricating step-by-step rationales that are not in the data.
    ans = clean_answer(answer)
    return f"The final answer is \\boxed{{{ans}}}."

example_prompt = build_user_prompt(sft_train_df.loc[0, "prompt"])
example_response = build_assistant_response(sft_train_df.loc[0, "answer"])
print(example_prompt[:1000])
print(example_response)


Instruction:
You are solving Alice's Wonderland rule-discovery puzzles. Infer the hidden transformation from the examples and answer the final query. End with the final answer in \boxed{...}.

Puzzle:
In Alice's Wonderland, numbers are secretly converted into a different numeral system. Some examples are given below:
8 -> VIII
5 -> V
97 -> XCVII
16 -> XVI
Now, write the number 95 in the Wonderland numeral system.

Response:

The final answer is \boxed{XCV}.


In [8]:
# Load Nemotron model and tokenizer, then attach LoRA

import site

# Same utility path used in the official demo notebook.
cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
if os.path.exists(cutlass_pkg_path):
    site.addsitedir(cutlass_pkg_path)

import kagglehub
import mamba_ssm
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
print("MODEL_PATH:", MODEL_PATH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=dtype,
)

# Important for training with gradient checkpointing.
model.config.use_cache = False
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

LORA_RANK = 32

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=64,
    target_modules=r".*\.(in_proj|out_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


MODEL_PATH: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 14,555,136 || all params: 31,592,492,480 || trainable%: 0.0461


In [9]:
# Dataset and data collator
# Labels are masked for the prompt tokens, so loss is only computed on the assistant answer.

from torch.utils.data import Dataset

class ReasoningSFTDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.items = []
        eos = tokenizer.eos_token or ""
        for _, row in df.iterrows():
            prompt_text = build_user_prompt(row["prompt"])
            answer_text = build_assistant_response(row["answer"]) + eos

            prompt_ids = tokenizer(prompt_text, add_special_tokens=False).input_ids
            answer_ids = tokenizer(answer_text, add_special_tokens=False).input_ids

            # Keep the answer intact. If needed, truncate the prompt from the left.
            max_prompt_len = max_length - len(answer_ids)
            if max_prompt_len <= 0:
                answer_ids = answer_ids[:max_length]
                prompt_ids = []
            elif len(prompt_ids) > max_prompt_len:
                prompt_ids = prompt_ids[-max_prompt_len:]

            input_ids = prompt_ids + answer_ids
            attention_mask = [1] * len(input_ids)
            labels = [-100] * len(prompt_ids) + answer_ids

            self.items.append({
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels,
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]


def make_collator(tokenizer):
    pad_id = tokenizer.pad_token_id
    def collate(features):
        max_len = max(len(f["input_ids"]) for f in features)
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for f in features:
            pad_len = max_len - len(f["input_ids"])
            batch["input_ids"].append(f["input_ids"] + [pad_id] * pad_len)
            batch["attention_mask"].append(f["attention_mask"] + [0] * pad_len)
            batch["labels"].append(f["labels"] + [-100] * pad_len)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}
    return collate

train_dataset = ReasoningSFTDataset(sft_train_df, tokenizer, MAX_LENGTH)
valid_dataset = ReasoningSFTDataset(sft_valid_df, tokenizer, MAX_LENGTH)
data_collator = make_collator(tokenizer)

print("Train examples:", len(train_dataset))
print("Validation examples:", len(valid_dataset))
print("First tokenized length:", len(train_dataset[0]["input_ids"]))


Train examples: 2300
Validation examples: 200
First tokenized length: 113


In [10]:
# Train the LoRA adapter

from transformers import Trainer, TrainingArguments

bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16 = torch.cuda.is_available() and not bf16

training_args = TrainingArguments(
    output_dir="/kaggle/working/train_logs",
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
    bf16=bf16,
    fp16=fp16,
    optim="adamw_torch",
    max_grad_norm=0.3,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

trainer.train()


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
10,8.820515
20,4.840181
30,4.375909
40,3.320824
50,3.387337
60,3.211287
70,2.547287
80,3.262366
90,2.858784
100,3.100276


TrainOutput(global_step=288, training_loss=3.1378308998213873, metrics={'train_runtime': 3680.2, 'train_samples_per_second': 0.625, 'train_steps_per_second': 0.078, 'total_flos': 7.59162395682793e+16, 'train_loss': 3.1378308998213873, 'epoch': 1.0})

In [11]:
import math

BOX_RE = re.compile(r"\\boxed\{([^{}]+)\}")

def extract_answer(text):
    matches = BOX_RE.findall(str(text))
    if matches:
        return matches[-1].strip()
    # Simple fallback: last non-empty line with common prefixes removed.
    lines = [ln.strip() for ln in str(text).splitlines() if ln.strip()]
    if not lines:
        return ""
    last = lines[-1]
    last = re.sub(r"^(final answer|answer)\s*[:=]\s*", "", last, flags=re.I)
    return last.strip().strip(".")

def normalize_answer(x):
    return str(x).strip()

def is_number(x):
    try:
        float(str(x).replace(",", ""))
        return True
    except Exception:
        return False

def is_correct(pred, truth):
    p = normalize_answer(pred)
    t = normalize_answer(truth)
    if p == t:
        return True
    if is_number(p) and is_number(t):
        return math.isclose(float(p.replace(",", "")), float(t.replace(",", "")), rel_tol=1e-6, abs_tol=1e-6)
    return False

@torch.no_grad()
def generate_one(prompt):
    model.eval()
    user_prompt = build_user_prompt(prompt)
    enc = tokenizer(user_prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    device = next(model.parameters()).device
    enc = {k: v.to(device) for k, v in enc.items()}
    out = model.generate(
        **enc,
        do_sample=False,
        max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    gen_ids = out[0][enc["input_ids"].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True)

if RUN_SMOKE_VALIDATION:
    rows = sft_valid_df.head(SMOKE_VALIDATION_ROWS).copy()
    records = []
    for _, row in rows.iterrows():
        raw = generate_one(row["prompt"])
        pred = extract_answer(raw)
        truth = normalize_answer(row["answer"])
        records.append({
            "id": row.get("id", ""),
            "truth": truth,
            "pred": pred,
            "correct": is_correct(pred, truth),
            "raw_output": raw[:500],
        })
    val_report = pd.DataFrame(records)
    display(val_report)
    print("Smoke validation accuracy:", val_report["correct"].mean() if len(val_report) else None)
else:
    print("Smoke validation skipped.")


,id,truth,pred,correct,raw_output
0,be877da5,02,02,True,The final answer is \boxed{02}.
1,675e4260,17.70,17.71,False,The final answer is \boxed{17.71}.
2,845fee60,01110000,10110001,False,The final answer is \boxed{10110001}.
3,6221b30e,12.56,12.31,False,The final answer is \boxed{12.31}.
4,94b7db05,teacher found the strange treasure,teacher draws the mysterious mirror,False,The final answer is \boxed{teacher draws the m...
5,3cbc472a,LXXIX,LXXIX,True,The final answer is \boxed{LXXIX}.
6,4a940571,hatter studies the wise castle,rabbit studies the wise message,False,The final answer is \boxed{rabbit studies the ...
7,c84c150e,00010001,00000010,False,The final answer is \boxed{00000010}.
8,1c4ac945,queen sees beyond village,queen sees beyond treasure,False,The final answer is \boxed{queen sees beyond t...
9,10b19ba1,62.82,62.5,False,The final answer is \boxed{62.5}.


Smoke validation accuracy: 0.3125


In [12]:
# Save adapter and package final submission.zip
# Do not zip the whole working directory. Put exactly the adapter files at the root of the zip.

Path(ADAPTER_DIR).mkdir(parents=True, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)

adapter_config_path = Path(ADAPTER_DIR) / "adapter_config.json"
adapter_model_path = Path(ADAPTER_DIR) / "adapter_model.safetensors"

assert adapter_config_path.exists(), f"Missing {adapter_config_path}"
assert adapter_model_path.exists(), f"Missing {adapter_model_path}"

with open(adapter_config_path, "r") as f:
    cfg = json.load(f)

rank = cfg.get("r", None)
print("Adapter rank:", rank)
assert rank is None or int(rank) <= 32, "Adapter rank must be <= 32"

if os.path.exists(SUBMISSION_ZIP):
    os.remove(SUBMISSION_ZIP)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(adapter_config_path, arcname="adapter_config.json")
    z.write(adapter_model_path, arcname="adapter_model.safetensors")

with zipfile.ZipFile(SUBMISSION_ZIP, "r") as z:
    names = z.namelist()
    print("Zip contents:", names)
    assert "adapter_config.json" in names
    assert "adapter_model.safetensors" in names

print("Created:", SUBMISSION_ZIP)
print("Size MB:", os.path.getsize(SUBMISSION_ZIP) / 1e6)


Adapter rank: 32
Zip contents: ['adapter_config.json', 'adapter_model.safetensors']
Created: /kaggle/working/submission.zip
Size MB: 40.955144
